In [1]:
import requests
import pandas as pd
import numpy as np
import time
import os

In [2]:
ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

In [6]:
stock_returns = pd.read_csv("stock_returns.csv")
etf_returns = pd.read_csv("etf_returns.csv")

stock_returns["date"] = pd.to_datetime(stock_returns["date"])
stock_returns = stock_returns.set_index("date").sort_index()

etf_returns["date"] = pd.to_datetime(etf_returns["date"])
etf_returns = etf_returns.set_index("date").sort_index()

tickers = stock_returns.columns.tolist() + etf_returns.columns.tolist()
tickers

['AAPL',
 'AMZN',
 'CAT',
 'JNJ',
 'JPM',
 'KO',
 'MSFT',
 'NVDA',
 'V',
 'XOM',
 'EEM',
 'EFA',
 'IWM',
 'QQQ',
 'SPY',
 'TLT',
 'VNQ',
 'XLE',
 'XLK',
 'XLV']

In [10]:
# Use your existing Paper API keys and tickers list
HEADERS = {
    "accept": "application/json",
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": SECRET_KEY
}

def pull_decade_news_requests(ticker_list):
    all_news = []
    symbols_str = ",".join(ticker_list)
    next_page_token = None
    
    print(f"Fetching news for {len(ticker_list)} tickers (2015-2025)...")

    while True:
        # Construct the URL with pagination
        url = "https://data.alpaca.markets/v1beta1/news"
        params = {
            "start": "2015-01-01T00:00:00Z",
            "end": "2025-12-31T23:59:59Z",
            "symbols": symbols_str,
            "limit": 50,  # Max per request
            "sort": "desc",
            "page_token": next_page_token
        }

        response = requests.get(url, headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Error: {response.status_code} - {response.text}")
            break

        data = response.json()
        articles = data.get("news", [])
        
        if not articles:
            break

        # Extract the fields you need for your model
        for item in articles:
            all_news.append({
                "timestamp": item["created_at"],
                "symbols": item["symbols"],
                "headline": item["headline"],
                "summary": item["summary"]
            })

        # Update the token for the next page
        next_page_token = data.get("next_page_token")
        
        # Log progress so you know it's working
        if len(all_news) % 500 == 0:
            print(f"Progress: {len(all_news)} articles collected...")

        # Stop if no more pages exist
        if not next_page_token:
            break

        # Essential for Paper/Free tier (200 requests/min limit)
        time.sleep(0.3)

    return pd.DataFrame(all_news)

In [11]:
news_df = pull_decade_news_requests(tickers)

Fetching news for 20 tickers (2015-2025)...
Progress: 500 articles collected...
Progress: 1000 articles collected...
Progress: 1500 articles collected...
Progress: 2000 articles collected...
Progress: 2500 articles collected...
Progress: 3000 articles collected...
Progress: 3500 articles collected...
Progress: 4000 articles collected...
Progress: 4500 articles collected...
Progress: 5000 articles collected...
Progress: 5500 articles collected...
Progress: 6000 articles collected...
Progress: 6500 articles collected...
Progress: 7000 articles collected...
Progress: 7500 articles collected...
Progress: 8000 articles collected...
Progress: 8500 articles collected...
Progress: 9000 articles collected...
Progress: 9500 articles collected...
Progress: 10000 articles collected...
Progress: 10500 articles collected...
Progress: 11000 articles collected...
Progress: 11500 articles collected...
Progress: 12000 articles collected...
Progress: 12500 articles collected...
Progress: 13000 articles c

In [12]:
news_df.to_csv("Market_news_data.csv", index=False)